<a href="https://colab.research.google.com/github/werowe/HypatiaAcademy/blob/master/ml/euro_random_forest_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Random Forest Regression: Predicting the Euro % Change

This notebook matches the example used in the slides.

We will predict the **percentage change in the euro tomorrow** using simple economic features:

- `interest_rates_up` — are interest rates increasing?
- `trade_deficit_shrinking` — is the trade deficit shrinking?
- `inflation_high` — is inflation high?
- `usd_strength_high` — is the U.S. dollar strong?
- `euro_change_pct` — the target value we want to predict

This is a **regression** problem because the output is a number, not a category.


---
## 1. Load Libraries

We use:

- `pandas` to create and inspect the dataset
- `scikit-learn` to train the models
- `matplotlib` to visualize the results


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error


---
## 2. Create a Small Dataset

This is a toy dataset for teaching.

Each row represents one market situation.

The target column, `euro_change_pct`, is the percentage change in the euro the next day.

For example, `1.4` means the euro increased by 1.4 percent.  
`-0.6` means the euro decreased by 0.6 percent.


In [ ]:
# Toy data that matches the regression example in the slides.
# 1 means True / Yes. 0 means False / No.

data = {
    "interest_rates_up":        [1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0],
    "trade_deficit_shrinking":  [1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1],
    "inflation_high":           [0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1],
    "usd_strength_high":        [0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1],
    "euro_change_pct":          [1.4, 1.0, 0.2, -0.3, 0.8, -0.2, 0.1, -0.9, 0.7, 1.2, 0.4, -1.0, 0.9, -0.1, 0.3, 0.2, 0.5, -0.6]
}

df = pd.DataFrame(data)
df


---
## 3. Define Features and Target

The features are the input columns.

The target is the numeric value we want to predict.


In [ ]:
# X contains the input features.
X = df[[
    "interest_rates_up",
    "trade_deficit_shrinking",
    "inflation_high",
    "usd_strength_high"
]]

# y contains the target value: tomorrow's euro percentage change.
y = df["euro_change_pct"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)


---
## 4. Train/Test Split

We split the data into training and test sets.

The model learns from the training data.

Then we test it on data it has not seen before.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])


---
## 5. Decision Tree Regressor

A regression tree asks yes/no questions about the features.

But the leaf nodes contain numbers, not class labels.

For example, one leaf might predict `+1.4%`, while another leaf might predict `-0.6%`.


In [ ]:
tree_model = DecisionTreeRegressor(
    max_depth=3,
    random_state=42
)

tree_model.fit(X_train, y_train)

tree_preds = tree_model.predict(X_test)

print("Decision Tree predictions:")
print(np.round(tree_preds, 2))


### Visualize the Decision Tree

This shows the questions the tree learned.

Each internal node is a split.

Each leaf contains a predicted percentage change.


In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree_model,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    impurity=False
)
plt.title("Decision Tree Regressor: Predict Euro % Change")
plt.show()


---
## 6. Random Forest Regressor

A random forest trains many decision trees.

Each tree sees a different bootstrap sample of the data.

At each split, each tree also sees only a random subset of the features.

This makes the trees different from each other.

For regression, the forest averages the numeric predictions from all trees.


In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

print("Random Forest predictions:")
print(np.round(rf_preds, 2))


---
## 7. Compare with Linear Regression

Linear regression tries to fit one straight-line relationship.

A random forest can model nonlinear patterns because it combines many local decision rules.


In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
linear_preds = linear_model.predict(X_test)

results = pd.DataFrame({
    "actual": y_test.values,
    "linear_regression": linear_preds,
    "decision_tree": tree_preds,
    "random_forest": rf_preds
})

results.round(2)


---
## 8. Evaluate the Models

We use two common regression metrics:

- **R²**: how much variance the model explains
- **MAE**: mean absolute error, or the average size of the prediction error

Because this is a very small toy dataset, the scores are only for demonstration.


In [ ]:
def evaluate_model(name, y_true, preds):
    r2 = r2_score(y_true, preds)
    mae = mean_absolute_error(y_true, preds)
    print(f"{name}")
    print(f"  R²:  {r2:.3f}")
    print(f"  MAE: {mae:.3f} percentage points")
    print()


evaluate_model("Linear Regression", y_test, linear_preds)
evaluate_model("Decision Tree", y_test, tree_preds)
evaluate_model("Random Forest", y_test, rf_preds)


---
## 9. Actual vs Predicted Values

A perfect model would put every point on the diagonal line.

Points closer to the line mean better predictions.


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf_preds, alpha=0.8, edgecolors="white", linewidth=0.5)

min_val = min(y_test.min(), rf_preds.min())
max_val = max(y_test.max(), rf_preds.max())
plt.plot([min_val, max_val], [min_val, max_val], "--", label="Perfect prediction")

plt.xlabel("Actual Euro Change (%)")
plt.ylabel("Predicted Euro Change (%)")
plt.title("Random Forest: Actual vs Predicted Euro % Change")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---
## 10. Feature Importance

Random forests can estimate which features were most useful for prediction.

Higher importance means the feature contributed more to the model's splits.


In [ ]:
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.gca().invert_yaxis()
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


---
## Key Idea

For classification, a random forest uses voting.

For regression, a random forest averages numeric predictions.

In this notebook, each tree predicts a euro percentage change.

The forest averages those predictions to produce one final forecast.
